In [0]:
"""Phase3 · SDP shared config as an importable module.

Declarative Pipelines ignore `%run`, so pipeline source notebooks import this
module instead of running `00_config` as a notebook.

**Source of truth:** `config/uc_environment.yaml` (next to this repo root).

**Optional pipeline Configuration overrides** (win over YAML when set):
- `clinical.repo_root` — workspace path that contains `config/uc_environment.yaml`
- `clinical.uc_yaml_path` — full path to the YAML (optional)
- `clinical.stream_source` — `event_hubs` | `volume`
- any `clinical.*` / `phase3.*` path or Event Hubs key
"""

# from __future__ import annotations

import json
import sys
from pathlib import Path

__all__ = [
    "repo_root",
    "catalog",
    "env",
    "domain",
    "stream_source",
    "raw_volume_path",
    "raw_inbox_path",
    "autoloader_schema_path",
    "autoloader_listing_interval",
    "eh_hub",
    "eh_consumer_group",
    "eh_secret_scope",
    "eh_secret_key",
    "starting_position",
    "max_events_per_trigger",
    "event_type_filter",
    "ref_schema",
    "v_fhir_mapping_current",
    "v_fhir_resource_mapping_current",
    "event_hubs_options",
    "describe",
]


def _active_spark():
    from pyspark.sql import SparkSession
    return SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()


def _active_dbutils(spark):
    try:
        from databricks.sdk.runtime import dbutils as sdk_dbutils

        return sdk_dbutils
    except Exception:
        pass
    try:
        from pyspark.dbutils import DBUtils

        return DBUtils(spark)
    except Exception:
        return None


_spark = _active_spark()
_dbutils = _active_dbutils(_spark)


def _conf(*keys: str, default: str = "") -> str:
    for key in keys:
        try:
            val = _spark.conf.get(key)
        except Exception:
            val = None
        if val is not None and str(val).strip() != "":
            return str(val).strip()
    return default


def _workspace_path(path: str) -> str:
    p = path.strip().rstrip("/")
    if not p:
        return p
    if p.startswith("/Workspace/") or p.startswith("/Volumes/") or p.startswith("/dbfs/"):
        return p
    if p.startswith("/Repos/") or p.startswith("/Users/") or p.startswith("/Shared/"):
        return f"/Workspace{p}"
    return p


def _candidate_repo_roots() -> list[str]:
    roots: list[str] = []
    conf_root = _conf("clinical.repo_root", "phase3.repo_root", default="")
    if conf_root:
        roots.append(_workspace_path(conf_root))
    # This module lives at <repo_root>/SDP/sdp_config.py
    try:
        roots.append(str(Path(__file__).resolve().parents[1]).replace("\\", "/"))
    except NameError:
        pass
    roots.append("/Users/anto2003.sfn@gmail.com/databricks-code-repo/bs-db-usecases/rpm")
    seen: set[str] = set()
    out: list[str] = []
    for r in roots:
        r = r.rstrip("/")
        if r and r not in seen:
            seen.add(r)
            out.append(r)
    return out


def _resolve_yaml_path() -> str:
    explicit = _conf("clinical.uc_yaml_path", "rpm.uc_yaml_path", default="")
    if explicit:
        return _workspace_path(explicit)
    for root in _candidate_repo_roots():
        path = f"{root}/config/uc_environment.yaml"
        if Path(path).is_file():
            return path
    return f"{_candidate_repo_roots()[0]}/config/uc_environment.yaml"


def _load_yaml_cfg() -> dict:
    yaml_path = _resolve_yaml_path()
    print('yaml_path',yaml_path)
    repo_for_import = str(Path(yaml_path).parent.parent)
    print(repo_for_import,'-repo_for_import')
    if repo_for_import not in sys.path:
        sys.path.insert(0, repo_for_import)
    try:
        from config.uc_naming import load_uc_yaml

        cfg = load_uc_yaml(yaml_path)
        print(f"[Phase3 SDP] loaded yaml={yaml_path}")
        return cfg if isinstance(cfg, dict) else {}
    except Exception as exc:
        print(f"[Phase3 SDP] yaml not loaded ({exc}); using conf/defaults only")
        return {}


_yaml = _load_yaml_cfg()
# _sdp = _yaml.get("sdp") or {}
# _eh = _yaml.get("event_hubs") or {}
# _ops = _yaml.get("ops") or {}

In [0]:
from typing import Any
def _minimal_yaml_load(path: str) -> dict[str, Any]:
    """Tiny subset parser for flat keys used when PyYAML is absent on the cluster."""
    out: dict[str, Any] = {}
    stack: list[tuple[int, dict[str, Any]]] = [(0, out)]
    with open(path, encoding="utf-8") as fh:
        for raw in fh:
            line = raw.split("#", 1)[0].rstrip()
            if not line.strip():
                continue
            indent = len(line) - len(line.lstrip(" "))
            key, _, val = line.lstrip().partition(":")
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            while stack and indent < stack[-1][0]:
                stack.pop()
            parent = stack[-1][1]
            if val == "":
                child: dict[str, Any] = {}
                parent[key] = child
                stack.append((indent + 2, child))
            elif val.lower() in ("true", "false"):
                parent[key] = val.lower() == "true"
            else:
                parent[key] = val
    return out

In [0]:
def load_uc_yaml(path: str) -> dict[str, Any]:
    """Load config/uc_environment.yaml when PyYAML is available; else minimal parse."""
    try:
        import yaml  # type: ignore
    except ImportError:
        return _minimal_yaml_load(path)
    with open(path, encoding="utf-8") as fh:
        data = yaml.safe_load(fh) or {}
    if not isinstance(data, dict):
        raise ValueError(f"Expected mapping in {path}")
    return data

In [0]:
load_uc_yaml("/Workspace/Users/anto2003.sfn@gmail.com/databricks-code-repo/bs-db-usecases/rpm/config/uc_environment.yaml")